In [ ]:
"""
Expand the Karaman kinase-compound interaction matrix into a long-format table.

Input format:
  Row 0: molecule names    (cols 0-1 empty, cols 2+ are molecule names)
  Row 1: molecule SMILES   (cols 0-1 empty, cols 2+ are SMILES)
  Row 2+: kinase rows      (col 0 = kinase name, col 1 = UniProt ID,
                            cols 2+ = bioactivity in nM, blank = >10000 nM)

Output columns:
  molecule_name, SMILES, protein_name, UniProt_ID, bioactivity, label
"""

import pandas as pd
import numpy as np

INPUT_PATH = "../dataset/Karaman_origin.csv"
OUTPUT_PATH = "../dataset/Karaman_expand_final.csv"

INACTIVE_FILL = 10000.0   # nM, used when a cell is blank
ACTIVE_THRESHOLD = 1000.0 # nM, <= this is "active"
ACTIVE_THRESHOLD_STRICT = 100.0

def main():
    # Read with no header so row indices match the file layout exactly.
    # encoding='utf-8-sig' strips the BOM that's on this file.
    df = pd.read_csv(INPUT_PATH, header=None, dtype=str, encoding="utf-8-sig")

    # --- Extract molecule metadata (first two rows, from col 2 onwards) ---
    molecule_names = df.iloc[0, 2:].tolist()
    molecule_smiles = df.iloc[1, 2:].tolist()
    assert len(molecule_names) == len(molecule_smiles), "molecule rows misaligned"

    # --- Extract kinase metadata + bioactivity matrix (row 2 onwards) ---
    kinase_block = df.iloc[2:, :].reset_index(drop=True)
    protein_names = kinase_block.iloc[:, 0]
    uniprot_ids = kinase_block.iloc[:, 1]
    activity_matrix = kinase_block.iloc[:, 2:].copy()

    # Attach protein metadata as regular columns, then melt.
    activity_matrix.columns = molecule_names
    activity_matrix.insert(0, "protein_name", protein_names.values)
    activity_matrix.insert(1, "UniProt_ID", uniprot_ids.values)

    # --- Melt to long format ---
    long = activity_matrix.melt(
        id_vars=["protein_name", "UniProt_ID"],
        var_name="molecule_name",
        value_name="bioactivity",
    )

    # --- Drop rows with missing UniProt (the 5 RPS6KA N-terminal domains) ---
    n_before = len(long)
    long = long[long["UniProt_ID"].notna() & (long["UniProt_ID"].str.strip() != "")]
    n_after = len(long)
    print(f"Removed {n_before - n_after} rows with missing UniProt ID "
          f"({n_before} -> {n_after})")

    # --- Attach SMILES ---
    smiles_map = dict(zip(molecule_names, molecule_smiles))
    long["SMILES"] = long["molecule_name"].map(smiles_map)

    # --- Coerce bioactivity to numeric; fill blanks with 10000 nM ---
    long["bioactivity"] = pd.to_numeric(long["bioactivity"], errors="coerce")
    long["bioactivity"] = long["bioactivity"].fillna(INACTIVE_FILL)

    # --- Binary label: 1 if bioactivity <= 1000 nM ---
    long["label"] = (long["bioactivity"] <= ACTIVE_THRESHOLD).astype(int)
    long["label_strict"] = (long["bioactivity"] <= ACTIVE_THRESHOLD_STRICT).astype(int)
    # --- Reorder columns ---
    long = long[["molecule_name", "SMILES", "protein_name", "UniProt_ID",
                 "bioactivity", "label","label_strict"]]

    # --- Save ---
    long.to_csv(OUTPUT_PATH, index=False)

    # --- Summary ---
    print(f"\nWrote {len(long)} rows to {OUTPUT_PATH}")
    print(f"  Unique molecules:       {long['molecule_name'].nunique()}")
    print(f"  Unique proteins (name): {long['protein_name'].nunique()}")
    print(f"  Unique UniProt IDs:     {long['UniProt_ID'].nunique()}")
    print(f"  Active   (label=1):     {(long['label']==1).sum()}")
    print(f"  Inactive (label=0):     {(long['label']==0).sum()}")
    print(f"  Active   (label_strict=1):     {(long['label_strict']==1).sum()}")
    print(f"  Inactive (label_strict=0):     {(long['label_strict']==0).sum()}")
    print(f"  Filled with {INACTIVE_FILL:.0f} nM: "
          f"{(long['bioactivity']==INACTIVE_FILL).sum()}")


if __name__ == "__main__":
    main()

Removed 0 rows with missing UniProt ID (10374 -> 10374)

Wrote 10374 rows to ./Karaman_expand_final.csv
  Unique molecules:       38
  Unique proteins (name): 273
  Unique UniProt IDs:     273
  Active   (label=1):     1419
  Inactive (label=0):     8955
  Active   (label_strict=1):     576
  Inactive (label_strict=0):     9798
  Filled with 10000 nM: 7963
